In [1]:
# phase2_multi_city.py
# Goal: Fetch weather data for 15 cities and store it cleanly

import requests
import json
import time    # we'll use this to avoid hammering the API too fast

# --- CONFIGURATION ---
API_KEY = "e365ed8fdcf3fbe95aba8de8c70d2bc5"
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
UNITS = "metric"

# --- CITIES LIST ---
# Mix of Indian cities + global cities = better analysis + better portfolio
CITIES = [
    # Indian cities
    "Lucknow", "Delhi", "Mumbai", "Bangalore", "Kolkata",
    "Chennai", "Hyderabad", "Jaipur", "Pune", "Ahmedabad",
    # Global cities
    "London", "New York", "Tokyo", "Dubai", "Sydney"
]

In [2]:
# --- FUNCTION TO FETCH ONE CITY ---
def fetch_weather(city):
    """
    Fetches weather data for a single city.
    Returns a clean dictionary, or None if something goes wrong.
    """
    params = {
        "q": city,
        "appid": API_KEY,
        "units": UNITS
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        # timeout=10 means "give up if no response in 10 seconds"

        if response.status_code == 200:
            data = response.json()

            # Extract only the fields we care about
            weather_info = {
                "city"        : data["name"],
                "country"     : data["sys"]["country"],
                "temperature" : data["main"]["temp"],
                "feels_like"  : data["main"]["feels_like"],
                "temp_min"    : data["main"]["temp_min"],
                "temp_max"    : data["main"]["temp_max"],
                "humidity"    : data["main"]["humidity"],
                "pressure"    : data["main"]["pressure"],
                "wind_speed"  : data["wind"]["speed"],
                "condition"   : data["weather"][0]["main"],
                "description" : data["weather"][0]["description"],
                "visibility"  : data.get("visibility", None),
                # data.get() is safer — returns None if key doesn't exist
            }
            return weather_info

        elif response.status_code == 404:
            print(f"  [SKIPPED] '{city}' not found — check spelling")
            return None

        elif response.status_code == 429:
            print(f"  [WAIT] Rate limit hit — waiting 60 seconds...")
            time.sleep(60)
            return None

        else:
            print(f"  [ERROR] {city} → status code {response.status_code}")
            return None

    except requests.exceptions.Timeout:
        print(f"  [TIMEOUT] {city} took too long to respond")
        return None

    except requests.exceptions.ConnectionError:
        print(f"  [CONNECTION ERROR] Check your internet connection")
        return None

In [3]:
# --- MAIN LOOP ---
all_weather_data = []   # this list will hold one dict per city

print("Starting data collection...\n")
print(f"{'City':<15} {'Temp':>6} {'Humidity':>9} {'Condition'}")
print("-" * 50)

for city in CITIES:
    result = fetch_weather(city)

    if result is not None:                      # only add if successful
        all_weather_data.append(result)

        # Print a live progress update
        print(f"{result['city']:<15} {result['temperature']:>5.1f}°C  "
              f"{result['humidity']:>6}%   {result['condition']}")

    time.sleep(1)   # wait 1 second between calls — be polite to the API!

print("-" * 50)
print(f"\nDone! Collected data for {len(all_weather_data)} cities.")

Starting data collection...

City              Temp  Humidity Condition
--------------------------------------------------
Lucknow          34.0°C      22%   Haze
Delhi            31.1°C      29%   Haze
Mumbai           33.0°C      52%   Smoke
Bengaluru        31.7°C      47%   Clear
Kolkata          32.0°C      70%   Haze
Chennai          32.6°C      64%   Clouds
Hyderabad        34.2°C      33%   Haze
Jaipur           32.6°C      18%   Haze
Pune             35.6°C      12%   Clear
Ahmedabad        31.0°C      35%   Smoke
London            3.9°C      88%   Clear
New York         19.7°C      64%   Clouds
Tokyo            23.9°C      49%   Clouds
Dubai            24.9°C      63%   Clouds
Sydney           23.5°C      50%   Clear
--------------------------------------------------

Done! Collected data for 15 cities.


In [4]:
# --- INSPECT THE DATA ---
print("\n--- PREVIEW OF COLLECTED DATA ---")
for entry in all_weather_data:
    print(entry)

print(f"\nTotal records collected: {len(all_weather_data)}")
print(f"Keys in each record: {list(all_weather_data[0].keys())}")


--- PREVIEW OF COLLECTED DATA ---
{'city': 'Lucknow', 'country': 'IN', 'temperature': 33.99, 'feels_like': 32.15, 'temp_min': 33.99, 'temp_max': 33.99, 'humidity': 22, 'pressure': 1008, 'wind_speed': 3.09, 'condition': 'Haze', 'description': 'haze', 'visibility': 5000}
{'city': 'Delhi', 'country': 'IN', 'temperature': 31.05, 'feels_like': 29.69, 'temp_min': 31.05, 'temp_max': 31.05, 'humidity': 29, 'pressure': 1009, 'wind_speed': 2.06, 'condition': 'Haze', 'description': 'haze', 'visibility': 3500}
{'city': 'Mumbai', 'country': 'IN', 'temperature': 32.99, 'feels_like': 36.86, 'temp_min': 30.94, 'temp_max': 32.99, 'humidity': 52, 'pressure': 1010, 'wind_speed': 3.09, 'condition': 'Smoke', 'description': 'smoke', 'visibility': 2500}
{'city': 'Bengaluru', 'country': 'IN', 'temperature': 31.69, 'feels_like': 33.15, 'temp_min': 30.57, 'temp_max': 33.06, 'humidity': 47, 'pressure': 1013, 'wind_speed': 3.13, 'condition': 'Clear', 'description': 'clear sky', 'visibility': 8000}
{'city': 'Kolk